# Wav2Vec2 Arabic -> OpenVINO CPU-Backend

Dieses Notebook exportiert `jonatasgrosman/wav2vec2-large-xlsr-53-arabic` direkt nach OpenVINO IR und startet danach das Live-Backend für die Mobile-App.

**Exportpfad:** Abhängigkeiten installieren, PyTorch-Referenz laden, OpenVINO-FP32-IR exportieren und mit Testaudio gegen PyTorch prüfen.

**Backendpfad:** Speicher freigeben, OpenVINO-Modell auf CPU laden, Audio-Preprocessing und GOP/LLR-Scoring definieren, FastAPI starten und den Tailscale-Funnel öffnen.

Der Backend-Forward läuft ausschließlich über OpenVINO auf der CPU; `torch` bleibt nur für Preprocessing und `forced_align` erforderlich. `OVModelForCTC(export=True)` exportiert direkt nach OpenVINO IR (`.xml`/`.bin`) ohne ONNX-Zwischenschritt.

## Schritt 1 — Installation

Installiert OpenVINO, NNCF, Audio-Verarbeitung und die Backend-Abhängigkeiten.
Nach der Installation den Kernel manuell über **Kernel -> Restart Kernel** neu starten,
damit neue NumPy-/OpenVINO-Bibliotheken sauber geladen werden. Danach ab Schritt 2
weiterarbeiten; Schritt 1 nicht erneut ausführen.

In [ ]:
# Installation: OpenVINO-Export + Live-Backend fuer die Mobile-App.
# Mit sys.executable -m pip funktioniert die Zelle in Colab und VS Code.
import importlib.util
import os
import shutil
import subprocess
import sys

PIP_PACKAGES = [
    "optimum-intel[openvino]", "transformers", "soundfile", "librosa",
    "gTTS", "pydub", "nest_asyncio", "python-multipart", "fastapi",
    "uvicorn[standard]", "silero-vad",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PIP_PACKAGES], check=True)

if shutil.which("ffmpeg") is None:
    if os.path.exists("/content") and shutil.which("apt-get"):
        subprocess.run(["apt-get", "-qq", "install", "-y", "ffmpeg"], check=True)
    else:
        raise RuntimeError("ffmpeg fehlt. Bitte ffmpeg installieren und Run All erneut starten.")

print("Installierte Kernpakete:")
for mod in ("transformers", "openvino", "fastapi", "uvicorn", "silero_vad"):
    spec = importlib.util.find_spec(mod)
    print(f"  {mod:14s} {'OK' if spec else 'FEHLT'}")
print("✅ Abhaengigkeiten bereit. Kein automatischer Kernel-Neustart erforderlich.")

Installierte Kernpakete:
  transformers   OK
  openvino       OK
  nncf           OK
  fastapi        OK
  uvicorn        OK
  silero_vad     OK
✅ Abhaengigkeiten bereit. Kein automatischer Kernel-Neustart erforderlich.


## Schritt 2 — Versionen prüfen, PyTorch-Referenz laden

Die PyTorch-FP32-Ausgabe ist die Referenz für den späteren OpenVINO-FP32-CPU-Forward. `blank_id` (`config.pad_token_id`) und das Frame-Raster bleiben identisch zum Backend.

In [ ]:
import os
from importlib.metadata import PackageNotFoundError, version

import numpy as np, torch, transformers, openvino as ov, optimum.intel

# Laptop-freundlich: OpenVINO/torch nicht automatisch mit allen Kernen starten.
SAFE_THREADS = max(1, min(4, os.cpu_count() or 1))
torch.set_num_threads(SAFE_THREADS)

try:
    OPTIMUM_INTEL_VERSION = version("optimum-intel")
except PackageNotFoundError:
    OPTIMUM_INTEL_VERSION = "unbekannt"

print("transformers  ", transformers.__version__)
print("torch         ", torch.__version__)
print("openvino      ", ov.__version__)
print("optimum-intel ", OPTIMUM_INTEL_VERSION)
print("CPU-Device    ", ov.Core().get_property("CPU", "FULL_DEVICE_NAME"))

from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC

MODEL_ID    = "jonatasgrosman/wav2vec2-large-xlsr-53-arabic"
OV_FP32_DIR = "ov_wav2vec2_ar_fp32"

processor   = Wav2Vec2Processor.from_pretrained(MODEL_ID)
torch_model = Wav2Vec2ForCTC.from_pretrained(MODEL_ID).eval()

SR       = processor.feature_extractor.sampling_rate
BLANK_ID = torch_model.config.pad_token_id
VOCAB    = torch_model.config.vocab_size

print(f"\nSampleRate={SR}  vocab={VOCAB}  blank_id={BLANK_ID}")
print(f"Params={sum(p.numel() for p in torch_model.parameters())/1e6:.1f}M")
print(f"CPU-Threads={SAFE_THREADS}")

/home/alghobariw/Desktop/temp/app/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


transformers   5.5.4
torch          2.13.0+cu130
openvino       2026.3.0-22451-8a17657b995-releases/2026/3
nncf           3.3.0
optimum-intel  2.1.0
CPU-Device     13th Gen Intel(R) Core(TM) i7-13800H


Loading weights: 100%|██████████| 424/424 [00:00<00:00, 11214.43it/s]


SampleRate=16000  vocab=51  blank_id=0
Params=315.5M
CPU-Threads=4


## Schritt 3 — Export nach OpenVINO IR

`compile=False`: erst speichern, kompiliert wird später mit explizitem CPU-Config
(Schritt 6). Der `processor` **muss** mitgespeichert werden — ohne `vocab.json` /
`preprocessor_config.json` ist im Deployment kein Decoding möglich.

In [ ]:
from optimum.intel import OVModelForCTC

ov_model = OVModelForCTC.from_pretrained(MODEL_ID, export=True, compile=False)

ov_model.save_pretrained(OV_FP32_DIR)
processor.save_pretrained(OV_FP32_DIR)     # vocab/tokenizer/feature-extractor mitspeichern

!ls -la {OV_FP32_DIR}
print("\nIR-Groesse:")
!du -sh {OV_FP32_DIR}

# Shape-Signatur des IR pruefen (dynamische Sequenzlaenge = [1,?])
m = ov.Core().read_model(f"{OV_FP32_DIR}/openvino_model.xml")
for i in m.inputs:  print("IN :", i.get_any_name(), i.partial_shape)
for o in m.outputs: print("OUT:", o.get_any_name(), o.partial_shape)

## Schritt 4 — Testaudio und erste Inferenz

Das Testaudio wird per gTTS synthetisiert, damit der Backend-Pfad ohne zusätzliche
Dateien und ohne Datensatz-Authentifizierung getestet werden kann.

In [ ]:
import librosa
from gtts import gTTS

TEXT = "بسم الله الرحمن الرحيم"
gTTS(TEXT, lang="ar").save("sample_ar.mp3")
audio, _ = librosa.load("sample_ar.mp3", sr=SR, mono=True)
print(f"Audio: {len(audio)} samples = {len(audio)/SR:.2f}s")

def features(a):
    """Genau die Vorverarbeitung, die auch asr_app.py nutzt."""
    return processor(a, sampling_rate=SR, return_tensors="np",
                     padding=False).input_values.astype(np.float32)

def greedy(logits):
    ids = np.asarray(logits).argmax(-1)
    return processor.batch_decode(torch.as_tensor(ids))[0]

feats = features(audio)

# PyTorch-Referenz (FP32)
with torch.no_grad():
    logits_pt = torch_model(torch.from_numpy(feats)).logits.numpy()

# OpenVINO IR (FP32)
ov_fp32 = OVModelForCTC.from_pretrained(OV_FP32_DIR)
logits_ov = np.asarray(ov_fp32(torch.from_numpy(feats)).logits)

print("logits shape :", logits_pt.shape, "->", logits_pt.shape[1], "Frames a ~20ms")
print("PyTorch      :", greedy(logits_pt))
print("OpenVINO     :", greedy(logits_ov))

## Schritt 5 — FP32-Logit-Treue prüfen

Abnahmekriterien statt Transkript-Vergleich:

* **FP32-IR:** `max|dLogit| < 1e-3`, Top-1-Frame-Match `100 %`, `dGOP` praktisch `0`.
* Das ist die relevante Prüfung für den OpenVINO-CPU-Backend-Pfad.

In [ ]:
def log_softmax(x, axis=-1):
    m = x.max(axis=axis, keepdims=True)
    e = np.exp(x - m)
    return x - m - np.log(e.sum(axis=axis, keepdims=True))

def compare(ref, cand, name):
    assert ref.shape == cand.shape, f"Frame-Mismatch {ref.shape} vs {cand.shape}"
    lp_r, lp_c = log_softmax(ref), log_softmax(cand)
    d_logit    = np.abs(ref - cand)
    d_logprob  = np.abs(lp_r - lp_c)
    top1       = (ref.argmax(-1) == cand.argmax(-1)).mean()
    corr       = np.corrcoef(ref.ravel(), cand.ravel())[0, 1]
    # Effekt auf den tatsaechlich gescorten Wert: logprob des Greedy-Pfads
    idx   = ref.argmax(-1)[0]
    gop_r = lp_r[0, np.arange(len(idx)), idx].mean()
    gop_c = lp_c[0, np.arange(len(idx)), idx].mean()
    print(f"--- {name} ---")
    print(f"  max |dLogit|        : {d_logit.max():.4f}")
    print(f"  mean|dLogProb|      : {d_logprob.mean():.5f}   max: {d_logprob.max():.4f}")
    print(f"  Top-1-Frame-Match   : {top1*100:.2f} %")
    print(f"  Pearson r           : {corr:.6f}")
    print(f"  mean GOP (ref/cand) : {gop_r:.4f} / {gop_c:.4f}  -> d={abs(gop_r-gop_c):.4f}")
    print(f"  Transkript identisch: {greedy(ref) == greedy(cand)}")

compare(logits_pt, logits_ov, "OpenVINO FP32 vs PyTorch FP32")

---

# Live-Backend für die Mobile-App

Ab hier läuft das Backend mit dem OpenVINO-FP32-Modell aus Teil A. Der ASR-Forward
läuft auf der CPU; `torch` bleibt nur für `forced_align` und die bestehende
Preprocessing-/Scoring-Pipeline geladen.

Die Zellen werden in dieser Reihenfolge ausgeführt:

1. Teil-A-Speicher freigeben
2. Backend-Imports laden
3. OpenVINO-Modell und Silero-VAD laden
4. Audio-Preprocessing und Scoring definieren
5. FastAPI-App definieren
6. Tailscale-Funnel starten und URL ausgeben

In [ ]:
# Teil-A-Modelle freigeben, bevor das Backend sein eigenes OpenVINO-Modell kompiliert.
import gc
for _name in ("torch_model", "ov_model", "ov_fp32", "core"):
    globals().pop(_name, None)
gc.collect()

try:
    import psutil, os as _os
    print(f"RSS jetzt: {psutil.Process(_os.getpid()).memory_info().rss/1e9:.2f} GB")
except Exception:
    pass

In [ ]:
import io, os, re, time, subprocess, threading, urllib.request, unicodedata
from typing import List, Dict, Any

import numpy as np
import torch
import torchaudio.functional as AF
from pydub import AudioSegment
from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC
from silero_vad import load_silero_vad, get_speech_timestamps

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SR = 16000
print(f"✅ Device: {device}, torch {torch.__version__}")

## OpenVINO-CPU-Modell laden

Der ASR-Forward wird durch `OVCTCModel` auf OpenVINO ersetzt. Der aktive Pfad nutzt
bewusst das FP32-IR aus Schritt 3: auf einer CPU ist FP16 nicht automatisch schneller
und kann je nach Hardware intern wieder in FP32 umgewandelt werden.

Ein `threading.Lock` serialisiert die wiederverwendete `InferRequest`; das passt zum
Einzel-Request-Betrieb des Backends und verhindert parallele Zugriffe auf denselben
OpenVINO-Request.

In [ ]:
# ---- OpenVINO statt PyTorch: Drop-in-Ersatz fuer asr_model ----------------
import threading
from types import SimpleNamespace
import openvino as ov

# Fuer den CPU-Test bewusst den stabilen FP32-OpenVINO-IR verwenden.
VARIANT   = "fp32"
MODEL_DIR = OV_FP32_DIR
assert os.path.exists(f"{MODEL_DIR}/openvino_model.xml"), (
    f"{MODEL_DIR} fehlt - erst Schritt 3 ausfuehren.")

ASR_MODEL_ID = f"jonatasgrosman/wav2vec2-large-xlsr-53-arabic (OpenVINO {VARIANT.upper()})"

USE_FP16 = False
DTYPE    = torch.float32
device   = torch.device("cpu")

N_THREADS = max(1, min(4, os.cpu_count() or 1))
OV_RUNTIME_CONFIG = {
    "PERFORMANCE_HINT":      "LATENCY",
    "NUM_STREAMS":           "1",
    "INFERENCE_NUM_THREADS": str(N_THREADS),
    "CACHE_DIR":             "ov_cache",
}


class OVCTCModel:
    """Drop-in-Ersatz fuer Wav2Vec2ForCTC auf Basis eines OpenVINO-IR."""

    def __init__(self, model_dir: str, cfg: dict):
        core          = ov.Core()
        self.compiled = core.compile_model(f"{model_dir}/openvino_model.xml", "CPU", cfg)
        self.req      = self.compiled.create_infer_request()
        self._in      = self.compiled.input(0)
        self._out     = self.compiled.output(0)
        self._lock    = threading.Lock()
        self.config   = SimpleNamespace(pad_token_id=None)

    def __call__(self, input_values):
        x = input_values.detach().cpu().numpy().astype(np.float32)
        with self._lock:
            logits = self.req.infer({self._in: x})[self._out].copy()
        return SimpleNamespace(logits=torch.from_numpy(logits))

    def eval(self):
        return self


print(f"Lade OpenVINO-Modell aus {MODEL_DIR}  ({N_THREADS} Threads) …")
asr_processor = Wav2Vec2Processor.from_pretrained(MODEL_DIR)
asr_model     = OVCTCModel(MODEL_DIR, OV_RUNTIME_CONFIG)
ASR_VOCAB     = asr_processor.tokenizer.get_vocab()
ASR_BLANK_ID  = asr_processor.tokenizer.pad_token_id
asr_model.config.pad_token_id = ASR_BLANK_ID
print(f"  OK  {len(ASR_VOCAB)} tokens, blank={ASR_BLANK_ID}, variant={VARIANT}")

print("Lade Silero VAD …")
vad_model = load_silero_vad()
print("  OK  Silero VAD 5")

_t0 = time.perf_counter()
_ = asr_model(torch.zeros(1, 2 * SR)).logits
print(f"Modelle geladen und aufgewaermt ({(time.perf_counter()-_t0)*1e3:.0f} ms Warm-up-Forward).")

## Schritt 12 — Audio-Preprocessing und Scoring

Beide Zellen wörtlich aus `wav2vec2_arabic_pronunciation.ipynb` (Zellen 4 und 5).
Die Signalkette (`decode → HP 80 Hz → RMS-Norm → gentle_trim → Kontext-Pad`), das
GOP-/LLR-Scoring, `forced_align` und die Tajweed-Schwellen sind unverändert — nur
so sind die Ergebnisse mit dem GPU-Backend vergleichbar.

`run_asr` in der zweiten Zelle ruft `asr_model(...)` auf und trifft damit
automatisch den OpenVINO-Shim aus Schritt 11. **`forced_align` bleibt torch** —
nur der Modell-Forward wandert zu OpenVINO.

In [ ]:
from scipy.signal import butter, sosfiltfilt

# Signalkette vor dem ASR (alle Schritte sind linear, phasentreu oder nicht-neuronal
# -> beruehren die spektrale Signatur arabischer Gutturale/Emphatika/Frikative nicht):
#   decode  -> HP80Hz  -> RMS-Norm -> gentle_trim (VAD) -> Kontext-Pad
_HPF_SOS = butter(2, 80.0, btype="highpass", fs=SR, output="sos")

def decode_audio(raw: bytes) -> np.ndarray:
    """Beliebiges Audioformat -> 16 kHz mono float32."""
    seg = AudioSegment.from_file(io.BytesIO(raw))
    seg = seg.set_frame_rate(SR).set_channels(1).set_sample_width(2)
    return np.asarray(seg.get_array_of_samples(), dtype=np.float32) / 32768.0

def _highpass(audio: np.ndarray) -> np.ndarray:
    """Nullphasiger Butterworth @ 80 Hz: entfernt DC-Offset, Handling-Rumpeln,
    Netzbrummen (50/60 Hz). Liegt unterhalb jeder Sprachformantenergie."""
    return sosfiltfilt(_HPF_SOS, audio).astype(np.float32)

def _normalize_level(audio: np.ndarray, target_dbfs: float = -20.0) -> np.ndarray:
    """RMS-Normalisierung auf konsistentes Pegel -> Silero-VAD-Schwelle wird reproduzierbar,
    und wav2vec2s eingebautes do_normalize=True bekommt ein saubereres Zero-Mean/Unit-Var-Ziel."""
    rms = float(np.sqrt(np.mean(audio ** 2)))
    if rms < 1e-6:
        return audio
    gain = 10.0 ** ((target_dbfs - 20.0 * np.log10(rms)) / 20.0)
    out  = audio * gain
    peak = float(np.max(np.abs(out)))
    if peak > 0.99:
        out = out / peak * 0.99
    return out.astype(np.float32)

def gentle_trim(audio: np.ndarray, pad_ms: int = 120) -> np.ndarray:
    """Nur führende/nachlaufende lange Stille entfernen. Zwischenpausen bleiben."""
    segs = get_speech_timestamps(torch.from_numpy(audio), vad_model,
                                 sampling_rate=SR, threshold=0.35)
    if not segs:
        return audio
    pad = int(pad_ms * SR / 1000)
    start = max(0, segs[0]["start"] - pad)
    end   = min(len(audio), segs[-1]["end"] + pad)
    return audio[start:end]

def _pad_context(audio: np.ndarray, ms: int = 250) -> np.ndarray:
    """Wav2vec2-Transformer sieht pro Frame ein bidirektionales Kontextfenster (~200 ms).
    Kurze Woerter (2-3 Buchstaben) verlieren sonst am Anfang/Ende Kontextframes und werden
    systematisch schlechter erkannt. Silence-Padding kostet keine Latenz und keine Genauigkeit."""
    pad = np.zeros(int(ms * SR / 1000), dtype=np.float32)
    return np.concatenate([pad, audio, pad])

def preprocess(raw: bytes) -> np.ndarray:
    audio = decode_audio(raw)
    audio = _highpass(audio)
    audio = _normalize_level(audio)
    audio = gentle_trim(audio)
    audio = _pad_context(audio)
    return audio

In [ ]:
# Nur klassisches Tashkeel entfernen. Hamza-Formen (أ إ آ ؤ ئ) bleiben als eigene Buchstaben erhalten.
_TASHKEEL = set("ًٌٍَُِّْٰ")

def strip_diacritics(text: str) -> str:
    nfd = unicodedata.normalize("NFD", text)
    return unicodedata.normalize("NFC", "".join(c for c in nfd if c not in _TASHKEEL))

# Positionsabhaengige Aequivalenzen (Anfang/Ende) fuer Posterior-Bewertung.
_START_EQUIV = {ch: "اأإآ" for ch in "اأإآ"}
_END_EQUIV   = {"ة": "ةه", "ه": "هة",
                "ى": "ىيا", "ي": "يى"}

# Linguistisch belegte Verwechslungen fuer den LLR-Test.
# Quellen: Al-Ani (1970) "Arabic Phonology"; Newman (2013);
# Standard-DaF/L2-Arabisch-Fehlerkataloge; Kinder-L1-Erwerbsstudien.
_CONFUSABLES: Dict[str, str] = {
    "ت": "طثد",
    "ث": "تسذف",
    "ح": "هخع",
    "خ": "حغك",
    "د": "تضذ",
    "ذ": "دزثظ",
    "ر": "لغ",
    "ز": "ذسظ",
    "س": "صثزش",
    "ش": "سج",
    "ص": "سض",
    "ض": "دظص",
    "ط": "تضد",
    "ظ": "زذض",
    "ع": "ءأاه",
    "غ": "خقر",
    "ق": "كغخ",
    "ك": "قخج",
    "ل": "ر",
    "ه": "حة",
    "ء": "ع",
    "ج": "شك",
}

def _equiv_ids(ch: str, pos: int, total: int) -> List[int]:
    if pos == 0 and ch in _START_EQUIV:
        alts = _START_EQUIV[ch]
    elif pos == total - 1 and ch in _END_EQUIV:
        alts = _END_EQUIV[ch]
    else:
        alts = ch
    ids = [ASR_VOCAB[c] for c in alts if c in ASR_VOCAB]
    return ids or [ASR_VOCAB[ch]]

def _confuse_ids(ch: str) -> List[int]:
    alts = _CONFUSABLES.get(ch, "")
    return [ASR_VOCAB[c] for c in alts if c in ASR_VOCAB]

# Umkehr-Map: Token-ID -> Buchstabe, fuer error_hint.
_ID_TO_CHAR = {tid: c for c, tid in ASR_VOCAB.items()}

def encode_target(word: str) -> List[int]:
    ids: List[int] = []
    for ch in word:
        tid = ASR_VOCAB.get(ch)
        if tid is None:
            raise ValueError(f"Zeichen {ch!r} nicht im ASR-Vokabular.")
        ids.append(tid)
    return ids

@torch.inference_mode()
def run_asr(audio: np.ndarray):
    inputs = asr_processor(audio, sampling_rate=SR, return_tensors="pt", padding=True)
    input_values = inputs.input_values.to(device=device, dtype=DTYPE)
    logits = asr_model(input_values).logits
    # log_softmax stabil in fp32, torchaudio.forced_align verlangt float32 CPU.
    log_probs = torch.log_softmax(logits.float(), dim=-1).cpu()
    transcription = asr_processor.batch_decode(log_probs.argmax(dim=-1))[0]
    return log_probs, transcription

def _runs_of_non_blank(tokens: List[int]) -> List[List[int]]:
    runs: List[List[int]] = []
    current: List[int] = []
    last: int = -1
    for t, tok in enumerate(tokens):
        if tok == ASR_BLANK_ID:
            if current: runs.append(current); current = []
            last = -1
        elif tok != last:
            if current: runs.append(current)
            current = [t]; last = tok
        else:
            current.append(t)
    if current: runs.append(current)
    return runs

# Kalibrierungskonstante: LLR=0 -> 50, LLR=+1 -> ~88, LLR=-1 -> ~12.
_LLR_K = 2.0

def _sigmoid(x: float) -> float:
    return 1.0 / (1.0 + float(np.exp(-x)))

def gop_score(log_probs: torch.Tensor, target_word: str) -> List[Dict[str, Any]]:
    target_ids = encode_target(target_word)
    if not target_ids:
        return []
    if log_probs.shape[1] < len(target_ids):
        raise ValueError("Aufnahme zu kurz für dieses Wort.")
    targets = torch.tensor([target_ids], dtype=torch.int32)
    aligned, _ = AF.forced_align(log_probs, targets, blank=ASR_BLANK_ID)
    runs = _runs_of_non_blank(aligned[0].tolist())
    total_len = len(target_word)
    results: List[Dict[str, Any]] = []
    for i, ch in enumerate(target_word):
        if i >= len(runs):
            results.append({"label": ch, "score": 0.0, "confidence": 0.0,
                            "llr": -5.0, "error_hint": None})
            continue

        frames   = runs[i]
        lp_frame = log_probs[0, frames]  # [F, V]

        # 1) Posterior-Score (klassisches GOP, positionsbewusst).
        equiv_ids  = _equiv_ids(ch, i, total_len)
        target_lp  = lp_frame[:, equiv_ids].max(dim=-1).values.mean().item()
        post_score = float(np.clip((target_lp + 3.0) / 3.0 * 100, 0, 100))
        conf       = float(np.exp(target_lp))

        # 2) LLR gegen dokumentierte Verwechslungen (Anti-Modell).
        confuse_ids = _confuse_ids(ch)
        if confuse_ids:
            per_frame_conf = lp_frame[:, confuse_ids]
            best_conf_lp   = per_frame_conf.max(dim=-1).values.mean().item()
            llr            = target_lp - best_conf_lp
            llr_score      = _sigmoid(_LLR_K * llr) * 100.0
            # Nur melden wenn Verwechslung staerker als Ziel.
            if llr < 0:
                best_col   = int(per_frame_conf.mean(dim=0).argmax().item())
                hint_id    = confuse_ids[best_col]
                error_hint = _ID_TO_CHAR.get(hint_id)
            else:
                error_hint = None
        else:
            llr, llr_score, error_hint = 5.0, 100.0, None

        # 3) Kombination: 40 % Posterior + 60 % LLR (LLR ist informativer).
        final = 0.4 * post_score + 0.6 * llr_score
        results.append({
            "label": ch,
            "score": float(np.clip(final, 0, 100)),
            "confidence": conf,
            "llr": float(llr),
            "error_hint": error_hint,
        })
    return results

## Schritt 13 — FastAPI-App

Wörtlich aus `wav2vec2_arabic_pronunciation.ipynb` (Zelle 6): `/assess` (HTTP),
`/stream` (WebSocket, Wort- und Ayah-Modus), `/health`, `/logs`. Auth per
`API_TOKEN` — aus dem Colab-Secret `API_TOKEN`, sonst automatisch generiert und
in Schritt 14 ausgegeben.

In [ ]:
from fastapi import FastAPI, UploadFile, File, Form, HTTPException, WebSocket, WebSocketDisconnect, Depends
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import Optional, Iterator
import asyncio, json
from collections import deque
from datetime import datetime, timezone

MAX_AUDIO_BYTES     = 3 * 1024 * 1024
MAX_AYAH_AUDIO_BYTES = 8 * 1024 * 1024
MIN_SAMPLES     = int(0.15 * SR)

# Cloudflared Quick Tunnel killt WS-Verbindungen nach ~60-100s Inaktivitaet.
# Wir senden alle 20s einen App-Level-Ping, damit die Verbindung fuer die
# ganze Kind-Session offen bleibt (sonst Reconnect-Cost pro Wort ~500ms).
WS_KEEPALIVE_SEC = 20

# Zwischen Wort-Frames im Ayah-Stream: minimaler Delay, damit die UI
# das progressive Einfaerben visuell wahrnimmt statt "alles auf einmal".
WORD_STREAM_DELAY_SEC = 0.035

# ----- Timing-Log (In-Memory Ring + Datei, ueber GET /logs + !tail abrufbar) -----
_LOG_BUF: deque = deque(maxlen=200)
_LOG_FILE = "/content/backend.log"
try:
    # datei zuruecksetzen bei jedem Cell-Rerun
    open(_LOG_FILE, "w").close()
except Exception:
    _LOG_FILE = "/tmp/backend.log"
    try: open(_LOG_FILE, "w").close()
    except Exception: pass

def _log(event: str, **kv):
    """Struktur-Log: Colab-Cell + In-Memory-Ring + Datei /content/backend.log.
    Ansicht:  !tail -n 40 /content/backend.log   oder   <BACKEND_URL>/logs"""
    entry = {"ts": datetime.now(timezone.utc).isoformat(timespec="milliseconds").replace("+00:00", "Z"),
             "event": event, **kv}
    _LOG_BUF.append(entry)
    kv_str = " ".join(f"{k}={v}" for k, v in kv.items())
    line = f"[{entry['ts']}] {event}  {kv_str}"
    print(line, flush=True)
    try:
        with open(_LOG_FILE, "a") as fh:
            fh.write(line + "\n"); fh.flush()
    except Exception:
        pass

class Unit(BaseModel):
    label: str
    score: float
    confidence: float
    llr: Optional[float] = None
    error_hint: Optional[str] = None

class AssessResponse(BaseModel):
    target: str
    transcription: str
    units: List[Unit]
    total: float
    duration_ms: int

app = FastAPI(title="Arabic Pronunciation API", version="2.0.0-colab")
app.add_middleware(CORSMiddleware, allow_origins=["*"],
                   allow_methods=["*"], allow_headers=["*"])

# --- Auth ---------------------------------------------------------------
# API_TOKEN aus Colab-Secret oder auto-generiert. Muss auch im App-Setting
# (Settings -> "Auth-Token") gesetzt sein, sonst 401.
import secrets as _secrets
try:
    from google.colab import userdata as _ud
    API_TOKEN = _ud.get("API_TOKEN") or None
except Exception:
    API_TOKEN = None
if not API_TOKEN:
    API_TOKEN = os.environ.get("API_TOKEN") or _secrets.token_urlsafe(24)

from fastapi import Header, Query, status
from fastapi.responses import JSONResponse

def _require_token(header_token: Optional[str], query_token: Optional[str]) -> None:
    """Konstantzeit-Vergleich gegen API_TOKEN. Raises 401 bei Mismatch."""
    provided = header_token or query_token or ""
    if not provided or not _secrets.compare_digest(provided, API_TOKEN):
        raise HTTPException(status.HTTP_401_UNAUTHORIZED, "Ungueltiger oder fehlender API-Token.")

def _auth_dep(
    x_api_token: Optional[str] = Header(default=None, alias="X-API-Token"),
    token: Optional[str] = Query(default=None),
) -> None:
    _require_token(x_api_token, token)

# Middleware: jeder HTTP-Request wird geloggt (auch /health). Damit sieht man
# im /logs sofort, ob der Client ueberhaupt am Backend ankommt.
@app.middleware("http")
async def _request_logger(request, call_next):
    t0 = time.perf_counter()
    resp = await call_next(request)
    # /logs selbst NICHT loggen, sonst rekursive Flut beim Auto-Refresh.
    if not request.url.path.startswith("/logs"):
        _log("http",
             method=request.method,
             path=request.url.path,
             status=resp.status_code,
             ms=int((time.perf_counter() - t0) * 1000),
             client=request.client.host if request.client else "?")
    return resp

def _score_word(raw: bytes, target: str) -> Dict[str, Any]:
    """Synchrone Bewertungs-Pipeline. Wird sowohl vom HTTP- als auch vom WS-Endpoint aufgerufen,
    damit die Bewertungsqualitaet identisch bleibt."""
    if len(raw) > MAX_AUDIO_BYTES:
        raise HTTPException(413, f"Audio > {MAX_AUDIO_BYTES // 1024} KB.")
    if not raw:
        raise HTTPException(400, "Leere Audiodatei.")
    try:
        wav = preprocess(raw)
    except Exception as e:
        raise HTTPException(400, f"Audio ungültig: {e}")
    if wav.size < MIN_SAMPLES:
        raise HTTPException(400, "Aufnahme zu kurz.")
    target_clean = strip_diacritics(target)
    log_probs, transcription = run_asr(wav)
    units = gop_score(log_probs, target_clean)
    total = float(np.mean([u["score"] for u in units])) if units else 0.0
    return {
        "target": target_clean,
        "transcription": transcription,
        "units": units,
        "total": total,
    }

# ---------------------------------------------------------------------------
# Ayah-Modus: eine ganze Ayah (mehrere Woerter) in EINEM ASR-Forward-Pass
# scoren und pro Wort progressiv an den Client streamen.
#
# Design:
#   1. Alle Woerter der Ayah werden zu EINEM Buchstaben-Ziel konkateniert.
#      forced_align liefert damit ein globales, konsistentes Alignment ueber
#      die gesamte Rezitation - genauer als N unabhaengige Wort-Alignments,
#      weil Uebergaenge zwischen Woertern (Waslah, Idghaam) mitmodelliert werden.
#   2. Aus den Buchstaben-Runs werden per Wort-Grenzen-Map wieder Wort-Scores
#      aggregiert (Mittelwert der Buchstaben-Scores + Mindestwert-Penalty
#      falls einzelne Buchstaben stark abfallen).
#   3. Der WS-Handler yielded die Wort-Ergebnisse einzeln mit minimalem Delay,
#      damit der Client die Einfaerbung wie eine Live-Auswertung rendert.
#
# Woerter mit Tashkeel werden per strip_diacritics gecleant; Hamza-Formen
# bleiben erhalten (siehe _TASHKEEL). Wenn die Aufnahme zu kurz ist, um alle
# Buchstaben zu enthalten, bekommen fehlende Runs Score 0 - der Client kann
# das als "abgeschnitten" darstellen.
# ---------------------------------------------------------------------------

def _score_ayah_streamed(raw: bytes, ayah_text: str) -> Iterator[Dict[str, Any]]:
    if len(raw) > MAX_AYAH_AUDIO_BYTES:
        raise HTTPException(413, f"Audio > {MAX_AYAH_AUDIO_BYTES // 1024} KB.")
    if not raw:
        raise HTTPException(400, "Leere Audiodatei.")

    # 1) Ayah in Woerter zerlegen (Whitespace-basiert). Leere Tokens verwerfen.
    raw_words = [w for w in ayah_text.split() if w.strip()]
    if not raw_words:
        raise HTTPException(400, "Ayah-Text leer.")

    words_clean: List[str] = []
    for w in raw_words:
        cw = strip_diacritics(w)
        if cw:
            words_clean.append(cw)
    if not words_clean:
        raise HTTPException(400, "Ayah-Text enthaelt keine bewertbaren Zeichen.")

    # 2) Kompletter Buchstaben-Stream + Wort-Grenzen-Map [start, end).
    all_chars: List[str] = []
    word_spans: List[tuple] = []
    for w in words_clean:
        s = len(all_chars)
        all_chars.extend(list(w))
        word_spans.append((s, len(all_chars)))

    # 3) Ziel-IDs; unbekannte Zeichen -> Fehler mit klarer Meldung.
    target_ids: List[int] = []
    for ch in all_chars:
        tid = ASR_VOCAB.get(ch)
        if tid is None:
            raise HTTPException(400, f"Zeichen {ch!r} nicht im ASR-Vokabular.")
        target_ids.append(tid)

    # 4) Preprocess + ASR (ein Forward-Pass fuer die ganze Ayah).
    t_pre = time.perf_counter()
    try:
        wav = preprocess(raw)
    except Exception as e:
        raise HTTPException(400, f"Audio ungueltig: {e}")
    if wav.size < MIN_SAMPLES:
        raise HTTPException(400, "Aufnahme zu kurz.")
    dt_pre = int((time.perf_counter() - t_pre) * 1000)

    t_asr = time.perf_counter()
    log_probs, transcription = run_asr(wav)
    dt_asr = int((time.perf_counter() - t_asr) * 1000)

    if log_probs.shape[1] < len(target_ids):
        raise HTTPException(400, "Aufnahme zu kurz fuer diese Ayah.")

    t_align = time.perf_counter()
    targets = torch.tensor([target_ids], dtype=torch.int32)
    aligned, _ = AF.forced_align(log_probs, targets, blank=ASR_BLANK_ID)
    runs = _runs_of_non_blank(aligned[0].tolist())
    dt_align = int((time.perf_counter() - t_align) * 1000)

    # 5) Start-Frame: gibt dem Client die Wort-Anzahl + Transkription zur Anzeige.
    yield {
        "kind": "start",
        "words_count": len(words_clean),
        "transcription": transcription,
    }

    t_score = time.perf_counter()
    per_word_scores: List[float] = []
    for wi, (start_c, end_c) in enumerate(word_spans):
        word = words_clean[wi]
        word_len = end_c - start_c
        char_units: List[Dict[str, Any]] = []

        for local_i, char_global_i in enumerate(range(start_c, end_c)):
            ch = all_chars[char_global_i]
            if char_global_i >= len(runs):
                char_units.append({
                    "label": ch, "score": 0.0, "confidence": 0.0,
                    "llr": -5.0, "error_hint": None,
                })
                continue

            frames = runs[char_global_i]
            lp_frame = log_probs[0, frames]

            # Positionsbewusstsein: Anfang/Ende gilt PRO WORT, nicht pro Ayah.
            equiv_ids = _equiv_ids(ch, local_i, word_len)
            target_lp = lp_frame[:, equiv_ids].max(dim=-1).values.mean().item()
            post_score = float(np.clip((target_lp + 3.0) / 3.0 * 100, 0, 100))
            conf = float(np.exp(target_lp))

            confuse_ids = _confuse_ids(ch)
            if confuse_ids:
                per_frame_conf = lp_frame[:, confuse_ids]
                best_conf_lp = per_frame_conf.max(dim=-1).values.mean().item()
                llr = target_lp - best_conf_lp
                llr_score = _sigmoid(_LLR_K * llr) * 100.0
                if llr < 0:
                    best_col = int(per_frame_conf.mean(dim=0).argmax().item())
                    hint_id = confuse_ids[best_col]
                    error_hint = _ID_TO_CHAR.get(hint_id)
                else:
                    error_hint = None
            else:
                llr, llr_score, error_hint = 5.0, 100.0, None

            final = 0.4 * post_score + 0.6 * llr_score
            char_units.append({
                "label": ch,
                "score": float(np.clip(final, 0, 100)),
                "confidence": conf,
                "llr": float(llr),
                "error_hint": error_hint,
            })

        # Wort-Score: Mittelwert MIT Min-Penalty. Ein katastrophaler Buchstabe
        # zieht das Ergebnis staerker runter als reines Averaging, aber nicht
        # so hart, dass ein sonst gutes Wort komplett rot wird.
        char_scores = [u["score"] for u in char_units]
        mean_s = float(np.mean(char_scores)) if char_scores else 0.0
        min_s  = float(np.min(char_scores))  if char_scores else 0.0
        word_score = float(np.clip(0.75 * mean_s + 0.25 * min_s, 0, 100))
        per_word_scores.append(word_score)

        yield {
            "kind": "word",
            "word_idx": wi,
            "target": word,
            "score": word_score,
            "units": char_units,
        }

    total = float(np.mean(per_word_scores)) if per_word_scores else 0.0
    dt_score = int((time.perf_counter() - t_score) * 1000)
    yield {
        "kind": "done",
        "total": total,
        "words_count": len(words_clean),
        "timings": {
            "audio_bytes": len(raw),
            "audio_samples": int(wav.size),
            "audio_ms": int(wav.size * 1000 / SR),
            "preprocess_ms": dt_pre,
            "asr_ms": dt_asr,
            "align_ms": dt_align,
            "score_ms": dt_score,
        },
    }

@app.get("/health")
def health():
    return {"status": "ok", "device": str(device),
            "asr_model": ASR_MODEL_ID, "vad": "Silero VAD 5",
            "fp16": USE_FP16,
            "endpoints": ["/assess (HTTP)", "/stream (WebSocket)", "/logs"]}

@app.get("/logs", dependencies=[Depends(_auth_dep)])
def get_logs(n: int = 50, fmt: str = "html"):
    """Letzte N Log-Eintraege. Aus dem Handy-Browser:  <BACKEND_URL>/logs?n=40
    HTML mit Auto-Refresh (Standard) oder ?fmt=json fuer maschinell."""
    n = max(1, min(int(n), _LOG_BUF.maxlen or 200))
    entries = list(_LOG_BUF)[-n:][::-1]  # neueste oben
    if fmt == "json":
        return {"count": len(entries), "entries": entries}
    from fastapi.responses import HTMLResponse
    if not entries:
        rows = "<tr><td colspan='2' style='color:#94a3b8'>Noch keine Requests aufgezeichnet.</td></tr>"
    else:
        keys = ["ts", "event"] + sorted({k for e in entries for k in e if k not in ("ts", "event")})
        head = "".join(f"<th>{k}</th>" for k in keys)
        body_rows = []
        for e in entries:
            cells = "".join(f"<td>{e.get(k, '')}</td>" for k in keys)
            body_rows.append(f"<tr>{cells}</tr>")
        rows = f"<tr>{head}</tr>" + "".join(body_rows)
    html = f"""<!doctype html><html><head><meta charset='utf-8'>
<meta name='viewport' content='width=device-width,initial-scale=1'>
<meta http-equiv='refresh' content='2'>
<title>Backend-Logs</title>
<style>
body{{font-family:-apple-system,Segoe UI,Roboto,sans-serif;margin:0;padding:12px;background:#0f172a;color:#e2e8f0}}
h1{{font-size:15px;margin:0 0 8px}}
table{{width:100%;border-collapse:collapse;font-size:11px;font-family:ui-monospace,Menlo,Consolas,monospace}}
th{{text-align:left;padding:6px 8px;background:#1e293b;color:#93c5fd;position:sticky;top:0}}
td{{padding:5px 8px;border-top:1px solid #1e293b;color:#e2e8f0;white-space:nowrap}}
tr:nth-child(even) td{{background:#0b1220}}
.small{{color:#64748b;font-size:11px}}
</style></head><body>
<h1>Backend-Logs <span class='small'>· auto-refresh 2s · n={n}</span></h1>
<table>{rows}</table>
</body></html>"""
    return HTMLResponse(content=html)

@app.post("/assess", response_model=AssessResponse, dependencies=[Depends(_auth_dep)])
def assess(audio: UploadFile = File(...), target: str = Form(...)):
    target = target.strip()
    if not target:
        raise HTTPException(400, "Zielwort fehlt.")
    t0 = time.perf_counter()
    raw = audio.file.read(MAX_AUDIO_BYTES + 1)
    try:
        result = _score_word(raw, target)
    except ValueError as e:
        raise HTTPException(400, str(e))
    result["duration_ms"] = int((time.perf_counter() - t0) * 1000)
    return AssessResponse(units=[Unit(**u) for u in result["units"]], **{
        k: v for k, v in result.items() if k != "units"
    })

@app.websocket("/stream")
async def stream_ws(ws: WebSocket):
    # Auth via ?token=... (mobile-Client sendet den Token so).
    _q_token = ws.query_params.get("token") or ws.headers.get("x-api-token")
    if not _q_token or not _secrets.compare_digest(_q_token, API_TOKEN):
        await ws.close(code=1008, reason="invalid token")
        _log("ws_reject", reason="bad_token", client=ws.client.host if ws.client else "?")
        return
    """Persistente Session pro Kind. Zwei Modi ueber dasselbe WS.

    A) Einzelwort-Modus (bestehend, Play-Modus):
       1. Client -> Server: {"target": "kitab"} (Text-Frame)
       2. Client -> Server: Binaer-Frame mit Audio
       3. Server -> Client: AssessResponse-JSON

    B) Ayah-Modus (Quran-Reading, progressives Per-Wort-Streaming):
       1. Client -> Server: {"mode": "ayah", "ayah": "bism allah..."}
       2. Client -> Server: Binaer-Frame mit Audio
       3. Server -> Client (Serie):
            {"kind":"start","words_count":N,"transcription":"..."}
            {"kind":"word","word_idx":0,"target":"...","score":..,"units":[..]}
            ...
            {"kind":"done","total":..,"duration_ms":..}

    Zusaetzlich sendet der Server alle WS_KEEPALIVE_SEC Sekunden
    {"ping": true}, damit Cloudflared die Verbindung nicht als idle killt.
    """
    await ws.accept()
    _log("ws_open", client=ws.client.host if ws.client else "?")

    async def keepalive():
        try:
            while True:
                await asyncio.sleep(WS_KEEPALIVE_SEC)
                await ws.send_json({"ping": True})
        except Exception:
            return

    ka_task = asyncio.create_task(keepalive())

    try:
        while True:
            ctrl = json.loads(await ws.receive_text())
            mode = str(ctrl.get("mode", "word")).lower()

            if mode == "ayah":
                ayah = str(ctrl.get("ayah", "")).strip()
                if not ayah:
                    await ws.send_json({"error": "Ayah-Text fehlt."})
                    continue
                t_ctrl = time.perf_counter()
                msg = await ws.receive()
                if "bytes" not in msg or msg["bytes"] is None:
                    await ws.send_json({"error": "Erwartete Binaerdaten (Audio)."})
                    continue
                raw: bytes = msg["bytes"]
                dt_bytes_ms = int((time.perf_counter() - t_ctrl) * 1000)
                t0 = time.perf_counter()
                try:
                    frames = await asyncio.to_thread(
                        lambda: list(_score_ayah_streamed(raw, ayah))
                    )
                except HTTPException as e:
                    _log("ayah_err", detail=e.detail, bytes=len(raw))
                    await ws.send_json({"error": e.detail}); continue
                except ValueError as e:
                    _log("ayah_err", detail=str(e), bytes=len(raw))
                    await ws.send_json({"error": str(e)}); continue
                except Exception as e:
                    _log("ayah_err", detail=str(e), bytes=len(raw))
                    await ws.send_json({"error": f"Serverfehler: {e}"}); continue

                dt_compute = int((time.perf_counter() - t0) * 1000)
                t_stream = time.perf_counter()
                for f in frames:
                    if f.get("kind") == "done":
                        f["duration_ms"] = int((time.perf_counter() - t0) * 1000)
                        f.setdefault("timings", {})["bytes_recv_ms"] = dt_bytes_ms
                    await ws.send_json(f)
                    if f.get("kind") == "word":
                        await asyncio.sleep(WORD_STREAM_DELAY_SEC)
                dt_stream = int((time.perf_counter() - t_stream) * 1000)

                # Zusammenfassung fuer /logs und Colab-Cell.
                done = next((f for f in frames if f.get("kind") == "done"), {})
                t = done.get("timings", {})
                _log("ayah",
                     words=done.get("words_count"),
                     total=round(done.get("total", 0), 1),
                     bytes=len(raw),
                     audio_ms=t.get("audio_ms"),
                     recv=dt_bytes_ms,
                     pre=t.get("preprocess_ms"),
                     asr=t.get("asr_ms"),
                     align=t.get("align_ms"),
                     score=t.get("score_ms"),
                     stream=dt_stream,
                     compute=dt_compute)
                continue

            # --- Einzelwort-Modus (backwards compatible) ---
            target = str(ctrl.get("target", "")).strip()
            if not target:
                await ws.send_json({"error": "Zielwort fehlt."})
                continue
            msg = await ws.receive()
            if "bytes" not in msg or msg["bytes"] is None:
                await ws.send_json({"error": "Erwartete Binaerdaten (Audio)."})
                continue
            raw: bytes = msg["bytes"]
            t0 = time.perf_counter()
            try:
                result = await asyncio.to_thread(_score_word, raw, target)
            except HTTPException as e:
                await ws.send_json({"error": e.detail})
                continue
            except ValueError as e:
                await ws.send_json({"error": str(e)})
                continue
            except Exception as e:
                await ws.send_json({"error": f"Serverfehler: {e}"})
                continue
            result["duration_ms"] = int((time.perf_counter() - t0) * 1000)
            await ws.send_json(result)
    except WebSocketDisconnect:
        _log("ws_close", reason="disconnect")
        return
    except Exception as e:
        _log("ws_close", reason=f"exception:{e}")
        try: await ws.send_json({"error": f"Serverfehler: {e}"})
        except Exception: pass
    finally:
        ka_task.cancel()

print("✅ API definiert:  GET /health   POST /assess   WS /stream  (Wort+Ayah, Keep-Alive 20s)")

## Schritt 14 — Tailscale-Funnel und Server starten

Wörtlich aus `wav2vec2_arabic_pronunciation.ipynb` (Zelle 7).

**Voraussetzungen** (einmalig im Tailscale-Admin):

1. Colab-Secret **`TS_AUTHKEY`** setzen → https://login.tailscale.com/admin/settings/keys
2. **HTTPS aktivieren** → https://login.tailscale.com/admin/dns → *Enable HTTPS*
3. **Funnel-Attribut** in den ACLs → https://login.tailscale.com/admin/acls
   ```json
   "nodeAttrs": [ { "target": ["*"], "attr": ["funnel"] } ]
   ```
4. Optional Colab-Secret **`API_TOKEN`** — sonst wird einer generiert und unten
   ausgegeben.

Der erste `tailscale cert`-Aufruf kann bis zu 180 s dauern. Läuft die Zelle in
einen Timeout: einfach **nochmal ausführen**, der Rest ist idempotent.

Am Ende stehen `Backend-URL`, `WebSocket-URL` und `Token` in der Ausgabe — die
trägst du im Settings-Screen der App ein.

In [ ]:
# ---- FastAPI + Tailscale Funnel -----------------------------------------
# Idempotent: einen vorhandenen tailscaled-Dienst weiterverwenden und nicht
# auf dem Laptop ungefragt beenden. TS_AUTHKEY aus Colab-Secret oder Umgebung.
import json, os, shutil, subprocess, sys, threading, time as _t, urllib.request
import uvicorn

PORT = 8000
IS_COLAB = "google.colab" in sys.modules or os.path.isdir("/content")

_key = os.environ.get("TS_AUTHKEY", "").strip()
if not _key and IS_COLAB:
    try:
        from google.colab import userdata
        _key = (userdata.get("TS_AUTHKEY") or "").strip()
    except Exception:
        pass
if not _key:
    raise RuntimeError(
        "TS_AUTHKEY fehlt. In Colab als Secret TS_AUTHKEY setzen oder lokal "
        "als Umgebungsvariable exportieren; der Key wird nicht im Notebook gespeichert."
    )

if shutil.which("tailscale") is None:
    if IS_COLAB:
        print("📦 Installiere Tailscale …")
        subprocess.run(
            "curl -fsSL https://tailscale.com/install.sh | sh",
            shell=True, check=True, stdout=subprocess.DEVNULL,
            stderr=subprocess.STDOUT,
        )
    else:
        raise RuntimeError("tailscale fehlt. Tailscale installieren und Run All erneut starten.")

# Auf dem Laptop/bei einem bereits laufenden Colab-Dienst nichts killen.
def _tailscale_status():
    try:
        return json.loads(subprocess.check_output(
            ["tailscale", "status", "--json"], stderr=subprocess.STDOUT, timeout=10
        ))
    except Exception:
        return None

status_json = _tailscale_status()
if status_json is None and IS_COLAB:
    os.makedirs("/var/run/tailscale", exist_ok=True)
    os.makedirs("/var/lib/tailscale", exist_ok=True)
    subprocess.Popen(
        ["tailscaled", "--tun=userspace-networking",
         "--socks5-server=localhost:1055",
         "--state=/var/lib/tailscale/tailscaled.state",
         "--socket=/var/run/tailscale/tailscaled.sock"],
        stdout=open("/tmp/tailscaled.log", "a"), stderr=subprocess.STDOUT,
    )
    for _ in range(30):
        if os.path.exists("/var/run/tailscale/tailscaled.sock"):
            break
        _t.sleep(0.5)
    else:
        raise RuntimeError("tailscaled-Socket nicht erreichbar; /tmp/tailscaled.log pruefen.")

# Auth-Key wird nur als Prozessargument an tailscale uebergeben und nie ausgegeben.
r = subprocess.run(
    ["tailscale", "up", f"--auth-key={_key}",
     "--hostname=colab-asr", "--accept-routes=false", "--timeout=30s"],
    capture_output=True, text=True, timeout=60,
)
if r.returncode != 0:
    raise RuntimeError(f"tailscale up fehlgeschlagen: {(r.stderr or r.stdout).strip()[:500]}")

_self = {}
for _ in range(60):
    status_json = _tailscale_status() or {}
    _self = status_json.get("Self") or {}
    if _self.get("Online") is True:
        break
    _t.sleep(1)
else:
    raise RuntimeError(
        f"Tailscale ist nach 60s offline: hostname={_self.get('HostName')} "
        f"online={_self.get('Online')}"
    )
print(f"✅ Tailscale online: {_self.get('HostName')} ({_self.get('DNSName', '').rstrip('.')})")

# FastAPI nur starten, wenn Port 8000 noch keinen gesunden Server hat.
def _fastapi_up():
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/health", timeout=2) as response:
            return response.status == 200
    except Exception:
        return False

if not _fastapi_up():
    threading.Thread(
        target=lambda: uvicorn.run(app, host="0.0.0.0", port=PORT,
                                   log_level="warning", access_log=False),
        daemon=True,
    ).start()
    for _ in range(30):
        if _fastapi_up():
            break
        _t.sleep(0.5)
    else:
        raise RuntimeError("FastAPI konnte auf Port 8000 nicht gestartet werden.")
print(f"✅ FastAPI bereit auf Port {PORT}")

DNS_NAME = (_self.get("DNSName") or "").rstrip(".")
if not DNS_NAME:
    raise RuntimeError("Tailscale-DNSName konnte nicht ermittelt werden.")
public_url = f"https://{DNS_NAME}"

# Funnel ist idempotent; bestehende Freigabe wird wiederverwendet.
funnel = subprocess.run(
    ["tailscale", "funnel", "--bg", str(PORT)],
    capture_output=True, text=True, timeout=120,
)
status = subprocess.run(
    ["tailscale", "funnel", "status"], capture_output=True, text=True, timeout=10,
)
status_text = (status.stdout or "") + (status.stderr or "")
if funnel.returncode != 0 and f":{PORT}" not in status_text and f"127.0.0.1:{PORT}" not in status_text:
    raise RuntimeError(
        "Tailscale Funnel konnte nicht aktiviert werden. "
        "HTTPS/Funnel-ACL im Tailscale-Admin pruefen.\n"
        + (funnel.stderr or funnel.stdout).strip()[:800]
    )
print("✅ Tailscale Funnel aktiv")

print("\n" + "=" * 68)
print(f"🌍 Backend-URL für die App:  {public_url}")
print(f"🔐 API-Token (in App eintragen):  {API_TOKEN}")
print("=" * 68)
print(f"Health-Check:  {public_url}/health")
print(f"WebSocket:     {public_url.replace('https://', 'wss://')}/stream?token={API_TOKEN}")
print(f"Live-Logs:     {public_url}/logs?token={API_TOKEN}")

## 🔎 Backend-Logs anschauen

Diese Zelle jederzeit **erneut ausführen**, um die letzten Backend-Timings zu sehen.  
Alternativ im Handy-Browser: `{PUBLIC_URL}/logs`  (HTML mit Auto-Refresh).


In [ ]:
# --- Backend-Logs Live (jederzeit erneut ausfuehren) ---
import subprocess
out = subprocess.run(["tail", "-n", "40", "/content/backend.log"], capture_output=True, text=True)
print(out.stdout or "(noch keine Logs)")
if 'public_url' in dir():
    print(f"\n🌍 Live im Browser: {public_url}/logs")


---

## Was beim Testen mit der App zu beobachten ist

* `asr_ms` in den Timings (`/logs` oder der Response) zeigt die CPU-Inferenzzeit.
* Die erste Anfrage einer neuen Audiolänge kann wegen dynamischer Shapes langsamer sein.
* Für reproduzierbare Tests dieselbe Audioaufnahme mehrfach verwenden.
* Die App verbindet sich über die am Ende ausgegebene HTTPS- und WebSocket-URL.

## Für die Übernahme ins Backend

* **Frame-Raster unverändert.** Der IR-Export ändert den Conv-Stack nicht; die ~20 ms/Frame und `blank_id = config.pad_token_id` gelten weiter.
* **`forced_align` braucht weiter torch/torchaudio.** Nur der ASR-Modell-Forward läuft über OpenVINO.
* **CPU-Laufzeit:** Das Modell wird mit `PERFORMANCE_HINT=LATENCY`, einem Stream, begrenzten Threads und `CACHE_DIR` kompiliert.
* **Bewertung:** Preprocessing, GOP-/LLR-Scoring und Tajweed-Schwellen bleiben identisch zum bestehenden Backend.